# LeetCode 1161 — Maximum Level Sum of a Binary Tree bằng Python

Link đề: https://leetcode.com/problems/maximum-level-sum-of-a-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def maxLevelSum(self, root: TreeNode | None) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên** — **mức (level) nhỏ nhất** có tổng giá trị nút lớn nhất. Gốc ở mức `1`. Điểm cốt lõi: duyệt **theo tầng** (BFS) hoặc DFS mang theo độ sâu để cộng dồn tổng từng mức; khi hoà, giữ mức **nhỏ hơn**. Lưu ý giá trị có thể **âm**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tính tổng mỗi mức của cây nhị phân, trả về mức nhỏ nhất đạt tổng lớn nhất.
2. Nắm công cụ: tự dựng `TreeNode` và hàm dựng cây từ mảng kiểu LeetCode, `collections.deque`, duyệt BFS theo tầng, DFS mang độ sâu (đệ quy và stack lặp).
3. Đi từ **baseline đo độ sâu từng nút** → **BFS theo tầng** → **DFS đệ quy với list tổng** → **DFS lặp bằng stack** → **BFS "hai list"**.
4. Hiểu bẫy: tổng có thể âm nên **không** khởi tạo max bằng `0`; hoà thì lấy mức nhỏ hơn (so sánh `>` chứ không `>=`).
5. Biết vì sao `level` bắt đầu từ `1`, không phải `0`.

## 2. Đề bài & ví dụ

Cho gốc `root` của một cây nhị phân. Gốc ở **mức 1**, các con của gốc ở mức 2, và cứ thế tiếp tục.

Trả về **mức `x` nhỏ nhất** sao cho tổng giá trị các nút ở mức `x` là **lớn nhất**.

**Ví dụ 1:** `root = [1,7,0,7,-8,null,null]` → `2`. Tổng mức 1 = `1`, mức 2 = `7 + 0 = 7`, mức 3 = `7 + (-8) = -1`.

**Ví dụ 2:** `root = [989,null,10250,98693,-89388,null,null,null,-32127]` → `2`.

**Ràng buộc:** số nút trong `[1, 10^4]`; `-10^5 <= Node.val <= 10^5`.

Mảng đầu vào ở dạng **level-order** của LeetCode: duyệt theo tầng, `null` nghĩa là vị trí trống; con của nút `null` không được ghi.

In [1]:
import random
import sys
import time
from collections import deque, defaultdict
from typing import List, Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(arr):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = trống)."""
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0])
    q = deque([root])
    i = 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i]); q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i]); q.append(node.right)
        i += 1
    return root


examples = [
    ([1, 7, 0, 7, -8, None, None], 2),
    ([989, None, 10250, 98693, -89388, None, None, None, -32127], 2),
]
for arr, want in examples:
    print(arr, "->", want)

[1, 7, 0, 7, -8, None, None] -> 2
[989, None, 10250, 98693, -89388, None, None, None, -32127] -> 2


## 3. Kiến thức nền

### 3.1 `TreeNode` và cây nhị phân

Mỗi nút có `val`, `left`, `right`. Cây nhị phân là cây mà mỗi nút có tối đa 2 con. **Mức** (level) của nút = số nút trên đường từ gốc đến nó (gốc = 1). **Độ sâu** (depth) thường đếm từ 0 — nên cẩn thận đổi `level = depth + 1`.

Để kiểm tra hàm `build_tree`, viết hàm ngược lại `to_list` (serialize lại theo tầng).

In [2]:
def to_list(root):
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

r = build_tree([1, 7, 0, 7, -8, None, None])
print(to_list(r))
assert to_list(r) == [1, 7, 0, 7, -8]
r2 = build_tree(examples[1][0])
print(to_list(r2))

[1, 7, 0, 7, -8]
[989, None, 10250, 98693, -89388, None, None, None, -32127]


### 3.2 `collections.deque` — hàng đợi hai đầu

`deque` cho `append` và `popleft` đều `O(1)`. Dùng `list.pop(0)` thì `O(n)` mỗi lần — tránh trong BFS.

In [3]:
q = deque([1, 2])
q.append(3)
print(q.popleft(), list(q))
assert list(q) == [2, 3]

1 [2, 3]


### 3.3 BFS theo tầng (level-order traversal)

Mẹo quan trọng: ở đầu mỗi tầng, `len(q)` chính là **số nút của tầng hiện tại**. Lặp đúng chừng ấy lần `popleft`, ta xử lý trọn một tầng, đồng thời các con vừa thêm thuộc tầng kế tiếp.

In [4]:
def levels(root):
    res, q = [], deque([root] if root else [])
    while q:
        layer = []
        for _ in range(len(q)):
            node = q.popleft()
            layer.append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        res.append(layer)
    return res

print(levels(build_tree([1, 7, 0, 7, -8])))
assert levels(build_tree([1, 7, 0, 7, -8])) == [[1], [7, 0], [7, -8]]

[[1], [7, 0], [7, -8]]


### 3.4 DFS mang theo độ sâu

DFS (Depth-First Search — tìm kiếm theo chiều sâu) đi hết một nhánh rồi mới quay lui. Nếu truyền thêm tham số `depth`, ta biết nút đang ở mức nào và có thể cộng vào `sums[depth]`. Ví dụ nhỏ: in `(val, depth)` theo thứ tự tiền thứ tự (preorder).

In [5]:
def preorder_with_depth(node, depth=0, out=None):
    if out is None:
        out = []
    if node:
        out.append((node.val, depth))
        preorder_with_depth(node.left, depth + 1, out)
        preorder_with_depth(node.right, depth + 1, out)
    return out

print(preorder_with_depth(build_tree([1, 7, 0, 7, -8])))

[(1, 0), (7, 1), (7, 2), (-8, 2), (0, 1)]


### 3.5 Chọn chỉ số của max với tie-break "nhỏ nhất"

`max(range(len(s)), key=s.__getitem__)` trả về chỉ số **đầu tiên** đạt max (Python `max` giữ phần tử đầu khi hoà). Khi tự viết vòng lặp, dùng `>` (nghiêm ngặt) để giữ mức nhỏ hơn.

In [6]:
s = [1, 7, -1, 7]
i1 = max(range(len(s)), key=s.__getitem__)
best_i, best = 0, float("-inf")
for i, v in enumerate(s):
    if v > best:           # '>' giữ chỉ số nhỏ nhất khi hoà
        best, best_i = v, i
print(i1, best_i)
assert i1 == best_i == 1

1 1


### 3.6 Khởi tạo max bằng `-inf`

Tổng mức có thể âm (ví dụ cây chỉ có `[-5]`). Nếu khởi tạo `best = 0`, ta sẽ không bao giờ cập nhật và trả về sai. Dùng `float('-inf')` hoặc tổng của mức 1.

In [7]:
print(float("-inf") < -10**18)
assert float("-inf") < -10**18

True


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2: `[989, null, 10250, 98693, -89388, null, null, null, -32127]`.

```
        989              mức 1: 989
           \
          10250          mức 2: 10250
          /    \
      98693   -89388     mức 3: 98693 + (-89388) = 9305
                 \
               -32127    mức 4: -32127
```

Tổng: `[989, 10250, 9305, -32127]` → max là `10250` ở mức **2**.

Vậy bài toán gồm hai bước tách biệt: (1) tính mảng `sums[level]`; (2) lấy chỉ số nhỏ nhất của max. Mọi lời giải chỉ khác ở bước (1).

In [8]:
print([sum(l) for l in levels(build_tree(examples[1][0]))])

[989, 10250, 9305, -32127]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng ngây thơ:** với mỗi mức `L = 1, 2, …`, duyệt **toàn bộ cây** để cộng các nút có mức `L`; dừng khi mức `L` không còn nút nào. Đây là cách "đúng theo định nghĩa", không cần nghĩ đến BFS.

In [9]:
def sum_at_level(node, target, level=1):
    """Tổng các nút ở mức target, và cờ cho biết có nút nào ở mức đó không."""
    if node is None:
        return 0, False
    if level == target:
        return node.val, True
    sl, fl = sum_at_level(node.left, target, level + 1)
    sr, fr = sum_at_level(node.right, target, level + 1)
    return sl + sr, fl or fr


def max_level_brute(root):
    best_level, best_sum = 1, float("-inf")
    L = 1
    while True:
        s, exists = sum_at_level(root, L)
        if not exists:
            break
        if s > best_sum:
            best_sum, best_level = s, L
        L += 1
    return best_level

for arr, want in examples:
    got = max_level_brute(build_tree(arr))
    print(got, want)
    assert got == want

2 2
2 2


**Độ phức tạp:** với mỗi mức ta duyệt tối đa cả cây → `O(n · h)` với `h` là chiều cao. Cây suy biến (dạng danh sách) có `h = n` → `O(n^2)` = `10^8` với `n = 10^4` — quá chậm trong Python. Đệ quy sâu `h` cũng có thể vượt giới hạn đệ quy mặc định (1000).

**Nút thắt:** mỗi nút bị thăm lại `h` lần. Ta cần cộng mỗi nút **đúng một lần** vào tổng của mức của nó.

## 6. Bước đột phá

> Mỗi nút chỉ đóng góp vào **một** mức duy nhất. Nếu khi thăm nút ta **biết mức của nó**, ta cộng ngay vào `sums[level]` — một lần duyệt là đủ.

Hai cách "biết mức" khi thăm nút:

- **BFS theo tầng:** xử lý cả tầng một lúc, mức = số tầng đã xử lý.
- **DFS mang depth:** truyền `depth + 1` xuống con.

Quy tắc "nếu… thì…":

- Nếu `depth == len(sums)` (lần đầu chạm mức này) thì `sums.append(val)`; ngược lại `sums[depth] += val`.
- Nếu `sums[i] > best` thì cập nhật (`>` để giữ mức nhỏ nhất khi hoà).

In [10]:
# Minh hoạ: cộng mỗi nút một lần bằng DFS mang depth
def level_sums_once(root):
    sums = []
    for val, d in preorder_with_depth(root):
        if d == len(sums):
            sums.append(val)
        else:
            sums[d] += val
    return sums

print(level_sums_once(build_tree(examples[1][0])))
assert level_sums_once(build_tree(examples[1][0])) == [989, 10250, 9305, -32127]

[989, 10250, 9305, -32127]


## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS theo tầng (lời giải chuẩn)

Dùng `deque`, mỗi vòng `while` xử lý một tầng, tính tổng tầng và so với max.

In [11]:
def max_level_bfs(root):
    best_level, best_sum = 1, float("-inf")
    q = deque([root])
    level = 0
    while q:
        level += 1
        layer_sum = 0
        for _ in range(len(q)):
            node = q.popleft()
            layer_sum += node.val
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        if layer_sum > best_sum:
            best_sum, best_level = layer_sum, level
    return best_level

for arr, want in examples:
    assert max_level_bfs(build_tree(arr)) == want
print("BFS OK")

BFS OK


**Độ phức tạp:** `O(n)` thời gian (mỗi nút vào/ra hàng đợi một lần); `O(w)` bộ nhớ với `w` là độ rộng lớn nhất của một tầng (tệ nhất `≈ n/2` với cây đầy đủ).

### 7.2 Cách 3 — DFS đệ quy với list tổng

Truyền `depth` xuống, cộng vào `sums`. Cuối cùng lấy chỉ số max. Ngắn gọn, nhưng cây suy biến sâu `10^4` sẽ vượt giới hạn đệ quy mặc định → cần `sys.setrecursionlimit` (ghi chú bên dưới) hoặc dùng cách 4.

In [12]:
def max_level_dfs_rec(root):
    sums = []

    def dfs(node, depth):
        if node is None:
            return
        if depth == len(sums):
            sums.append(0)
        sums[depth] += node.val
        dfs(node.left, depth + 1)
        dfs(node.right, depth + 1)

    dfs(root, 0)
    return max(range(len(sums)), key=sums.__getitem__) + 1   # +1 vì mức bắt đầu từ 1

for arr, want in examples:
    assert max_level_dfs_rec(build_tree(arr)) == want
print("DFS đệ quy OK")

DFS đệ quy OK


**Độ phức tạp:** `O(n)` thời gian, `O(h)` bộ nhớ cho ngăn xếp đệ quy + `O(h)` cho `sums`. Ghi chú: trên LeetCode cây sâu tới `10^4`; Python mặc định giới hạn ~1000 tầng đệ quy, nên khi nộp cách này cần `sys.setrecursionlimit(20000)`.

### 7.3 Cách 4 — DFS lặp bằng stack tường minh

Thay ngăn xếp gọi hàm bằng một `list` chứa cặp `(node, depth)`. Không lo giới hạn đệ quy.

In [13]:
def max_level_dfs_iter(root):
    sums = []
    stack = [(root, 0)]
    while stack:
        node, depth = stack.pop()
        if depth == len(sums):
            sums.append(0)
        sums[depth] += node.val
        if node.right:
            stack.append((node.right, depth + 1))
        if node.left:
            stack.append((node.left, depth + 1))
    best = 0
    for i in range(1, len(sums)):
        if sums[i] > sums[best]:
            best = i
    return best + 1

for arr, want in examples:
    assert max_level_dfs_iter(build_tree(arr)) == want
print("DFS lặp OK")

DFS lặp OK


Tại sao `depth == len(sums)` vẫn đúng với stack? Vì DFS luôn đi xuống từng bước một: để chạm tới độ sâu `d` lần đầu, ta phải đã thăm một nút ở độ sâu `d - 1`, nên `len(sums) ≥ d` luôn đúng khi pop nút ở độ sâu `d`.

### 7.4 Cách 5 — BFS "hai list" (không dùng deque)

Giữ `cur` là list các nút của tầng hiện tại; tầng sau là list các con. Rất "Pythonic", dễ đọc.

In [14]:
def max_level_two_lists(root):
    best_level, best_sum = 1, float("-inf")
    cur, level = [root], 1
    while cur:
        s = sum(node.val for node in cur)
        if s > best_sum:
            best_sum, best_level = s, level
        cur = [child for node in cur for child in (node.left, node.right) if child]
        level += 1
    return best_level

for arr, want in examples:
    assert max_level_two_lists(build_tree(arr)) == want
print("two lists OK")

two lists OK


**Độ phức tạp:** `O(n)` thời gian, `O(w)` bộ nhớ (hai list tầng liên tiếp).

## 8. TRACE từng bước

Trace BFS trên ví dụ 1 `[1, 7, 0, 7, -8]`:

| Tầng | Hàng đợi đầu tầng | Tổng tầng | best_sum | best_level |
|------|-------------------|-----------|----------|------------|
| 1 | [1] | 1 | 1 | 1 |
| 2 | [7, 0] | 7 | 7 | 2 |
| 3 | [7, -8] | -1 | 7 | 2 |

In [15]:
def trace_bfs(root):
    best_level, best_sum = 1, float("-inf")
    q = deque([root])
    level = 0
    while q:
        level += 1
        vals = [n.val for n in q]
        layer_sum = 0
        for _ in range(len(q)):
            node = q.popleft()
            layer_sum += node.val
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        updated = layer_sum > best_sum
        if updated:
            best_sum, best_level = layer_sum, level
        print(f"level={level} queue={vals} sum={layer_sum} {'UPDATE' if updated else ''} -> best=({best_level}, {best_sum})")
    return best_level

assert trace_bfs(build_tree([1, 7, 0, 7, -8])) == 2
print()
assert trace_bfs(build_tree(examples[1][0])) == 2

level=1 queue=[1] sum=1 UPDATE -> best=(1, 1)
level=2 queue=[7, 0] sum=7 UPDATE -> best=(2, 7)
level=3 queue=[7, -8] sum=-1  -> best=(2, 7)

level=1 queue=[989] sum=989 UPDATE -> best=(1, 989)
level=2 queue=[10250] sum=10250 UPDATE -> best=(2, 10250)
level=3 queue=[98693, -89388] sum=9305  -> best=(2, 10250)
level=4 queue=[-32127] sum=-32127  -> best=(2, 10250)


Trace DFS lặp: in stack và `sums` sau mỗi lần pop.

In [16]:
def trace_dfs_iter(root):
    sums, stack = [], [(root, 0)]
    while stack:
        node, depth = stack.pop()
        if depth == len(sums):
            sums.append(0)
        sums[depth] += node.val
        if node.right: stack.append((node.right, depth + 1))
        if node.left: stack.append((node.left, depth + 1))
        print(f"pop val={node.val:>4} depth={depth} sums={sums} stack={[(n.val, d) for n, d in stack]}")
    return sums

assert trace_dfs_iter(build_tree([1, 7, 0, 7, -8])) == [1, 7, -1]

pop val=   1 depth=0 sums=[1] stack=[(0, 1), (7, 1)]
pop val=   7 depth=1 sums=[1, 7] stack=[(0, 1), (-8, 2), (7, 2)]
pop val=   7 depth=2 sums=[1, 7, 7] stack=[(0, 1), (-8, 2)]
pop val=  -8 depth=2 sums=[1, 7, -1] stack=[(0, 1)]
pop val=   0 depth=1 sums=[1, 7, -1] stack=[]


## 9. Code nộp LeetCode

Chọn BFS theo tầng — không đệ quy, `O(n)`.

In [17]:
from collections import deque
from typing import Optional


class Solution:
    def maxLevelSum(self, root: Optional[TreeNode]) -> int:
        best_level, best_sum = 1, float("-inf")
        q = deque([root])
        level = 0
        while q:
            level += 1
            layer_sum = 0
            for _ in range(len(q)):
                node = q.popleft()
                layer_sum += node.val
                if node.left:
                    q.append(node.left)
                if node.right:
                    q.append(node.right)
            if layer_sum > best_sum:          # '>' giữ mức nhỏ nhất khi hoà
                best_sum, best_level = layer_sum, level
        return best_level


sol = Solution()
for arr, want in examples:
    assert sol.maxLevelSum(build_tree(arr)) == want
print("Solution OK")

Solution OK


(Trên LeetCode, chữ ký gốc `TreeNode | None` tương đương `Optional[TreeNode]`.)

## 10. Vì sao đúng

**Invariant của BFS:** ở đầu mỗi vòng `while`, hàng đợi chứa **chính xác** các nút của tầng `level + 1` (theo thứ tự trái → phải). Cơ sở: ban đầu hàng đợi chỉ có gốc (tầng 1). Bước quy nạp: ta pop đúng `len(q)` nút của tầng hiện tại, và chỉ đẩy vào các con của chúng — tất cả thuộc tầng kế tiếp, và mọi nút của tầng kế tiếp đều là con của một nút tầng hiện tại. Vậy `layer_sum` đúng bằng tổng tầng.

**Tie-break:** ta duyệt tầng theo thứ tự tăng dần và chỉ cập nhật khi tổng **lớn hơn hẳn**; nên khi nhiều tầng hoà max, tầng đầu tiên (nhỏ nhất) được giữ.

**Khởi tạo:** `best_sum = -inf` đảm bảo tầng 1 luôn được ghi nhận, kể cả khi tổng âm.

## 11. Time / Space Complexity

Gọi `n` = số nút, `h` = chiều cao, `w` = độ rộng tầng lớn nhất.

| Cách | Time | Space |
|------|------|-------|
| 1. Baseline từng mức | `O(n · h)` (tệ nhất `O(n^2)`) | `O(h)` đệ quy |
| 2. BFS deque | `O(n)` | `O(w)` |
| 3. DFS đệ quy | `O(n)` | `O(h)` |
| 4. DFS stack | `O(n)` | `O(h)` cho `sums` + stack tối đa `O(n)` tệ nhất |
| 5. BFS hai list | `O(n)` | `O(w)` |

Cây cân bằng: `h ≈ log n`, `w ≈ n/2`. Cây suy biến: `h = n`, `w = 1`. Vì vậy DFS tiết kiệm bộ nhớ hơn trên cây "mập", BFS tiết kiệm hơn trên cây "gầy".

## 12. Bộ test

### 12.1 Ví dụ + edge case

In [18]:
sys.setrecursionlimit(50000)   # cho các cách đệ quy trên cây sâu trong phần test
impls = {
    "brute": max_level_brute,
    "bfs": max_level_bfs,
    "dfs_rec": max_level_dfs_rec,
    "dfs_iter": max_level_dfs_iter,
    "two_lists": max_level_two_lists,
    "Solution": Solution().maxLevelSum,
}
edge = [
    ([5], 1),                          # một nút
    ([-5], 1),                         # một nút âm — bẫy khởi tạo 0
    ([-1, -2, -3], 1),                 # mọi tổng âm: -1 > -5
    ([0, 1, -1], 1),                   # hoà 0 = 0 → mức nhỏ hơn
    ([1, 2, 3, 4, None, None, -10], 2),
    ([-100, -200, -300, -20, -5, -10, None], 3),
    ([1, None, 2, None, 3, None, 4], 4),  # cây lệch phải
]
for name, f in impls.items():
    for arr, want in examples + edge:
        assert f(build_tree(arr)) == want, (name, arr)
print("all impls pass", len(examples + edge))

all impls pass 9


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

Sinh cây ngẫu nhiên bằng cách chèn nút vào vị trí trống ngẫu nhiên; giá trị nhỏ để hay hoà.

In [19]:
def random_tree(rng, n, lo=-3, hi=3):
    root = TreeNode(rng.randint(lo, hi))
    nodes = [root]
    for _ in range(n - 1):
        while True:
            p = rng.choice(nodes)
            side = rng.choice(["left", "right"])
            if getattr(p, side) is None:
                child = TreeNode(rng.randint(lo, hi))
                setattr(p, side, child)
                nodes.append(child)
                break
    return root

rng = random.Random(1161)
for _ in range(500):
    root = random_tree(rng, rng.randint(1, 30))
    want = max_level_brute(root)
    for name, f in impls.items():
        assert f(root) == want, name
print("random cross-check OK")

random cross-check OK


### 12.3 Test giới hạn

`n = 10^4`: cây lệch (sâu `10^4`) và cây gần đầy đủ.

In [20]:
def chain(n, rng):
    root = cur = TreeNode(rng.randint(-10**5, 10**5))
    for _ in range(n - 1):
        cur.right = TreeNode(rng.randint(-10**5, 10**5))
        cur = cur.right
    return root

rng = random.Random(99)
complete = build_tree([rng.randint(-10**5, 10**5) for _ in range(10**4)])
deep = chain(10**4, rng)
for root in (complete, deep):
    t0 = time.perf_counter()
    a = Solution().maxLevelSum(root)
    b = max_level_dfs_iter(root)
    c = max_level_two_lists(root)
    assert a == b == c
    print(a, f"{time.perf_counter() - t0:.3f}s")

13 0.003s
1103 0.007s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Khởi tạo `best_sum = 0`

In [21]:
def wrong_init_zero(root):
    best_level, best_sum = 1, 0
    for i, layer in enumerate(levels(root), 1):
        if sum(layer) > best_sum:
            best_sum, best_level = sum(layer), i
    return best_level

r = build_tree([-1, -2, -3])   # tổng: -1, -5 → đáp án 1 (đúng tình cờ)
r = build_tree([-9, -1, None]) # tổng: -9, -1 → đáp án 2
print("sai:", wrong_init_zero(r), "| đúng:", max_level_bfs(r))
assert wrong_init_zero(r) != max_level_bfs(r)

sai: 1 | đúng: 2


**Vì sao sai:** mọi tổng âm nên không bao giờ `> 0`; hàm trả về mức 1 mặc định. **Sửa:** `best_sum = float('-inf')`.

### Lỗi 2 — Dùng `>=` khi so sánh (lấy mức lớn nhất khi hoà)

In [22]:
def wrong_ge(root):
    best_level, best_sum = 1, float("-inf")
    for i, layer in enumerate(levels(root), 1):
        if sum(layer) >= best_sum:
            best_sum, best_level = sum(layer), i
    return best_level

r = build_tree([3, 1, 2])   # tổng 3, 3 → mức 1
print("sai:", wrong_ge(r), "| đúng:", max_level_bfs(r))
assert wrong_ge(r) == 2 and max_level_bfs(r) == 1

sai: 2 | đúng: 1


**Vì sao sai:** đề yêu cầu mức **nhỏ nhất**. **Sửa:** dùng `>`.

### Lỗi 3 — Trả về chỉ số 0-based

In [23]:
def wrong_zero_based(root):
    sums = [sum(l) for l in levels(root)]
    return max(range(len(sums)), key=sums.__getitem__)    # quên +1

r = build_tree([1, 7, 0, 7, -8])
print("sai:", wrong_zero_based(r), "| đúng:", max_level_bfs(r))
assert wrong_zero_based(r) == 1

sai: 1 | đúng: 2


**Vì sao sai:** gốc ở mức 1. **Sửa:** `+ 1`.

### Lỗi 4 — Không tách tầng trong BFS (dùng `while q` không có `for _ in range(len(q))`)

In [24]:
def wrong_no_layer(root):
    q = deque([root]); sums = []; level = 0
    while q:
        node = q.popleft()
        level += 1                     # tăng mức cho MỖI NÚT, không phải mỗi tầng
        sums.append(node.val)
        if node.left: q.append(node.left)
        if node.right: q.append(node.right)
    return max(range(len(sums)), key=sums.__getitem__) + 1

r = build_tree([1, 7, 0, 7, -8])
print("sai:", wrong_no_layer(r), "| đúng:", max_level_bfs(r))
r = build_tree([1, 4, 5])
print("sai:", wrong_no_layer(r), "| đúng:", max_level_bfs(r))
assert wrong_no_layer(r) != max_level_bfs(r)

sai: 2 | đúng: 2
sai: 3 | đúng: 2


**Vì sao sai:** mỗi nút bị coi là một mức riêng. Với `[1, 4, 5]`, tổng mức 2 là `9` nhưng code sai coi `5` là "mức 3". **Sửa:** chốt `len(q)` ở đầu tầng.

### Lỗi 5 — Đệ quy sâu không tăng giới hạn

Với cây lệch `10^4` nút, `dfs` đệ quy chạm `RecursionError` nếu giới hạn mặc định là 1000. Trong notebook đã `setrecursionlimit` ở mục 12; khi nộp, dùng BFS/stack hoặc tự tăng giới hạn.

## 14. Mẹo & nhận dạng pattern

- Đề nhắc tới **"level", "tầng", "mỗi hàng của cây"** → BFS theo tầng với mẹo `for _ in range(len(q))`.
- Cần **thống kê theo độ sâu** (tổng, max, trung bình, nút phải nhất) → hoặc BFS, hoặc DFS mang `depth` và list `res[depth]`.
- Có **giá trị âm** + "lớn nhất" → khởi tạo `-inf`.
- "Nhỏ nhất khi hoà" + duyệt tăng dần → so sánh nghiêm ngặt `>`.
- Cây có thể sâu `10^4` trong Python → ưu tiên lặp.

## 15. Biến thể

1. **LC 102 Binary Tree Level Order Traversal** — trả về list các tầng.
2. **LC 637 Average of Levels** — trung bình mỗi tầng (cẩn thận chia số nguyên).
3. **LC 515 Find Largest Value in Each Tree Row** — max mỗi tầng.
4. **LC 199 Binary Tree Right Side View** — nút cuối mỗi tầng.
5. **LC 2583 Kth Largest Sum in a Binary Tree** — tính tổng các tầng rồi lấy lớn thứ k.
6. Biến thể tự chế: trả về mức **lớn nhất** khi hoà, hoặc tổng **nhỏ nhất**.

In [25]:
# Biến thể 5: tổng tầng lớn thứ k (trả -1 nếu không đủ k tầng)
def kth_largest_level_sum(root, k):
    sums = [sum(l) for l in levels(root)]
    return sorted(sums, reverse=True)[k - 1] if k <= len(sums) else -1

r = build_tree([5, 8, 9, 2, 1, 3, 7, 4, 6])
print([sum(l) for l in levels(r)], kth_largest_level_sum(r, 2))
assert kth_largest_level_sum(r, 2) == 13

[5, 17, 13, 10] 13


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Baseline | Mỗi mức duyệt lại cả cây | `O(n·h)` | `O(h)` | Kiểm chứng |
| 2. BFS deque | Xử lý từng tầng, đếm `len(q)` | `O(n)` | `O(w)` | Lời giải chuẩn, nộp bài |
| 3. DFS đệ quy | `sums[depth] += val` | `O(n)` | `O(h)` | Code ngắn, cây không quá sâu |
| 4. DFS stack | Như 3 nhưng stack tường minh | `O(n)` | `O(h)`–`O(n)` | Cây sâu, không muốn đệ quy |
| 5. BFS hai list | `cur` → list con | `O(n)` | `O(w)` | Code gọn, dễ đọc |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `max_level_bfs(build_tree([0, -1, -1, 2, None, None, -2]))` = ?

**Bài 2 (tính tay):** viết tổng từng tầng của `[1, 2, 3, 4, 5, 6, 7, 8]` và đáp án.

**Bài 3 (điền TODO):** hàm trả về **list tổng từng tầng** bằng BFS.

**Bài 4 (tự viết lại):** viết lại cách 3 nhưng trả về mức **lớn nhất** khi hoà.

**Bài 5 (mini challenge):** trả về mức có **trung bình** giá trị lớn nhất (mức nhỏ nhất khi hoà), so sánh trung bình bằng nhân chéo để tránh số thực.

In [26]:
# Bài 3 — TODO
def level_sums_bfs(root):
    # TODO: trả về list tổng từng tầng, ví dụ [1, 7, -1]
    raise NotImplementedError

# assert level_sums_bfs(build_tree([1, 7, 0, 7, -8])) == [1, 7, -1]

In [27]:
# Bài 5 — TODO
def max_level_average(root):
    # TODO: so sánh s1/c1 > s2/c2 bằng s1*c2 > s2*c1
    pass

# assert max_level_average(build_tree([1, 7, 0, 7, -8])) == 2
# assert max_level_average(build_tree([4, 1, 9])) == 2   # 4 vs 5

## 18. Gợi ý 3 mức

- **Bài 1:** Mức 1: tầng 1 = 0. Mức 2: tầng 2 = −2, tầng 3 = 0. Mức 3: hoà 0 → mức nhỏ hơn.
- **Bài 2:** Mức 1: cây đầy đủ, tầng 3 = 4+5+6+7. Mức 2: tầng 4 = 8. Mức 3: tổng `[1, 5, 22, 8]`.
- **Bài 3:** Mức 1: copy cách 2. Mức 2: thay so sánh bằng `res.append(layer_sum)`. Mức 3: nhớ trường hợp `root` là `None` (trả `[]`).
- **Bài 4:** Mức 1: đổi `>` thành `>=`. Mức 2: hoặc duyệt `sums` từ cuối. Mức 3: `+1` cho chỉ số.
- **Bài 5:** Mức 1: lưu cặp `(sum, count)`. Mức 2: so sánh nhân chéo, count luôn dương. Mức 3: `>` nghiêm ngặt.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
assert max_level_bfs(build_tree([0, -1, -1, 2, None, None, -2])) == 1
# Bài 2
assert [sum(l) for l in levels(build_tree([1, 2, 3, 4, 5, 6, 7, 8]))] == [1, 5, 22, 8]
assert max_level_bfs(build_tree([1, 2, 3, 4, 5, 6, 7, 8])) == 3

# Bài 3
def level_sums_bfs_sol(root):
    res, q = [], deque([root] if root else [])
    while q:
        s = 0
        for _ in range(len(q)):
            node = q.popleft()
            s += node.val
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        res.append(s)
    return res

assert level_sums_bfs_sol(build_tree([1, 7, 0, 7, -8])) == [1, 7, -1]
assert level_sums_bfs_sol(None) == []

# Bài 4
def max_level_largest_tie(root):
    sums = level_sums_bfs_sol(root)
    best = 0
    for i in range(len(sums)):
        if sums[i] >= sums[best]:
            best = i
    return best + 1

assert max_level_largest_tie(build_tree([3, 1, 2])) == 2

# Bài 5
def max_level_average_sol(root):
    best_level, best = 1, None
    cur, level = [root], 1
    while cur:
        s, c = sum(n.val for n in cur), len(cur)
        if best is None or s * best[1] > best[0] * c:
            best, best_level = (s, c), level
        cur = [ch for n in cur for ch in (n.left, n.right) if ch]
        level += 1
    return best_level

assert max_level_average_sol(build_tree([1, 7, 0, 7, -8])) == 2
assert max_level_average_sol(build_tree([4, 1, 9])) == 2
assert max_level_average_sol(build_tree([5, 5, 5])) == 1
print("đáp án OK")

đáp án OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
best_level, best_sum, level = 1, float('-inf'), 0
q = deque([root])
while q:
    level += 1
    s = 0
    for _ in range(len(q)):        # chốt kích thước tầng
        node = q.popleft()
        s += node.val
        if node.left:  q.append(node.left)
        if node.right: q.append(node.right)
    if s > best_sum:               # '>' → mức nhỏ nhất khi hoà
        best_sum, best_level = s, level
return best_level
```

**Checklist:**

- [ ] `best_sum` khởi tạo `-inf` (không phải 0)?
- [ ] So sánh `>` để giữ mức nhỏ nhất?
- [ ] Mức bắt đầu từ **1**?
- [ ] Dùng `deque.popleft()` thay vì `list.pop(0)`?
- [ ] Chốt `len(q)` ở đầu tầng?
- [ ] Nếu dùng đệ quy: cây sâu `10^4` có vượt giới hạn không?